# Chapter 4 — Automated Backend Benchmarking Suite

**Applied Quantum Computing** | Lab 4 Optional Advanced

This notebook:
1. Queries all available IBM Quantum backends
2. Pulls live calibration data (T₁, T₂, gate errors)
3. Runs a standardized 3-qubit GHZ benchmark circuit
4. Computes a composite Quality Score for each backend
5. Generates an auto-formatted Backend Report Card

**Prerequisites:**
- IBM Quantum account (free tier works)
- Qiskit and Qiskit Runtime installed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/liquid-books/applied-quantum-computing/blob/main/notebooks/ch04-backend-benchmarking.ipynb)

In [ ]:
# Install dependencies (run once)
!pip install qiskit qiskit-ibm-runtime qiskit-aer pandas matplotlib seaborn --quiet

## Step 1: Authentication

Get your IBM Quantum API token from [quantum.ibm.com](https://quantum.ibm.com) → Account → API Token

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Option A: Save credentials once (run once, then use Option B)
# QiskitRuntimeService.save_account(channel='ibm_quantum', token='YOUR_TOKEN_HERE', overwrite=True)

# Option B: Load saved credentials
service = QiskitRuntimeService(channel='ibm_quantum')

print('Authentication successful!')
print(f'Available backends: {len(service.backends())}')

## Step 2: Query All Available Backends and Pull Calibration Data

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

def extract_backend_metrics(backend):
    """Extract key calibration metrics from a backend."""
    props = backend.properties()
    config = backend.configuration()
    
    if props is None:
        return None
    
    n_qubits = config.n_qubits
    
    # Collect T1 and T2 for all qubits
    t1_values = []
    t2_values = []
    readout_errors = []
    
    for q in range(n_qubits):
        try:
            t1 = props.t1(q)  # in seconds
            t2 = props.t2(q)  # in seconds
            re = props.readout_error(q)
            if t1 is not None: t1_values.append(t1 * 1e6)  # convert to microseconds
            if t2 is not None: t2_values.append(t2 * 1e6)
            if re is not None: readout_errors.append(re)
        except Exception:
            pass
    
    # Collect 2Q gate errors (CX/ECR gates)
    cx_errors = []
    for gate in props.gates:
        if gate.gate in ['cx', 'ecr', 'cz'] and len(gate.qubits) == 2:
            for param in gate.parameters:
                if param.name == 'gate_error':
                    cx_errors.append(param.value)
    
    return {
        'backend': backend.name,
        'n_qubits': n_qubits,
        'modality': 'Superconducting',  # All IBM backends are superconducting
        'T1_median_us': np.median(t1_values) if t1_values else np.nan,
        'T2_median_us': np.median(t2_values) if t2_values else np.nan,
        'T1_min_us': np.min(t1_values) if t1_values else np.nan,
        'T2_min_us': np.min(t2_values) if t2_values else np.nan,
        'cx_error_median': np.median(cx_errors) if cx_errors else np.nan,
        'cx_error_min': np.min(cx_errors) if cx_errors else np.nan,
        'readout_error_median': np.median(readout_errors) if readout_errors else np.nan,
        'n_cx_pairs': len(cx_errors),
        'last_update': props.last_update_date.strftime('%Y-%m-%d') if props.last_update_date else 'unknown'
    }

# Query all operational backends
print('Querying available backends...')
all_backends = service.backends(operational=True, simulator=False)

print(f'Found {len(all_backends)} operational backends. Extracting calibration data...')

backend_data = []
for backend in all_backends:
    print(f'  Processing {backend.name}...', end=' ')
    metrics = extract_backend_metrics(backend)
    if metrics:
        backend_data.append(metrics)
        print('✓')
    else:
        print('⚠ No calibration data')

df_backends = pd.DataFrame(backend_data)
print(f'\nSuccessfully processed {len(df_backends)} backends.')
df_backends.head()

## Step 3: Build the Standardized GHZ Benchmark Circuit

In [ ]:
from qiskit import QuantumCircuit
from qiskit.visualization import circuit_drawer

def build_ghz_circuit(n_qubits=3):
    """Build a standard n-qubit GHZ state circuit."""
    qc = QuantumCircuit(n_qubits, n_qubits)
    qc.h(0)
    for i in range(1, n_qubits):
        qc.cx(0, i)
    qc.measure(range(n_qubits), range(n_qubits))
    return qc

ghz3 = build_ghz_circuit(3)
print('3-qubit GHZ circuit:')
print(ghz3.draw(output='text'))
print()
print('Expected ideal distribution: 50% |000⟩, 50% |111⟩')
print('Any other outcome = error')

## Step 4: Run Benchmark on Available Backends

**Note:** This will use quantum credits. Each backend run uses a small number of shots. Reduce `n_backends_to_test` if you have limited credits.

In [ ]:
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
import time

SHOTS = 1024
N_BACKENDS_TO_TEST = 3  # Limit to save credits; increase for full benchmark

def compute_ghz_fidelity(counts, n_qubits=3):
    """Compute GHZ fidelity as fraction of ideal-state shots."""
    total_shots = sum(counts.values())
    ideal_states = {'0' * n_qubits, '1' * n_qubits}
    ideal_shots = sum(counts.get(s, 0) for s in ideal_states)
    return ideal_shots / total_shots

# Select backends to test (smallest qubit count first to minimize queue time)
test_backends_df = df_backends.nsmallest(N_BACKENDS_TO_TEST, 'n_qubits')
test_backend_names = test_backends_df['backend'].tolist()

print(f'Running GHZ benchmark on {N_BACKENDS_TO_TEST} backends:')
print(f'  Backends: {test_backend_names}')
print(f'  Shots per backend: {SHOTS}')
print()

ghz_results = {}

for backend_name in test_backend_names:
    print(f'Submitting job to {backend_name}...', end=' ')
    try:
        backend = service.backend(backend_name)
        
        # Transpile circuit for this backend
        pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
        transpiled = pm.run(ghz3)
        
        # Submit job
        sampler = Sampler(backend)
        job = sampler.run([transpiled], shots=SHOTS)
        print(f'Job ID: {job.job_id()}')
        
        # Wait for results
        result = job.result()
        counts = result[0].data.meas.get_counts()
        
        fidelity = compute_ghz_fidelity(counts)
        ghz_results[backend_name] = {
            'fidelity': fidelity,
            'counts': counts,
            'cx_depth': transpiled.count_ops().get('cx', 0) + transpiled.count_ops().get('ecr', 0)
        }
        print(f'  ✓ GHZ Fidelity: {fidelity:.3f}')
        
    except Exception as e:
        print(f'\n  ⚠ Error: {e}')
        ghz_results[backend_name] = {'fidelity': np.nan, 'counts': {}, 'cx_depth': 0}

print('\nBenchmark complete!')

## Step 5: Compute Quality Scores

**Quality Score Formula:**
$$QS = 0.35 \cdot \frac{T_2}{T_{2,\text{max}}} + 0.40 \cdot \left(1 - \frac{e_{2Q}}{e_{2Q,\text{max}}}\right) + 0.25 \cdot F_{GHZ}$$

In [ ]:
def compute_quality_scores(df, ghz_results):
    """Compute composite Quality Score for each backend."""
    df = df.copy()
    
    # Normalize T2 (higher is better)
    t2_max = df['T2_median_us'].max()
    df['T2_norm'] = df['T2_median_us'] / t2_max
    
    # Normalize 2Q error (lower is better → invert)
    err_max = df['cx_error_median'].max()
    df['fidelity_norm'] = 1 - (df['cx_error_median'] / err_max)
    
    # Add measured GHZ fidelity
    df['ghz_fidelity'] = df['backend'].map(
        {k: v.get('fidelity', np.nan) for k, v in ghz_results.items()}
    )
    # For backends without GHZ data, estimate from calibration
    df['ghz_fidelity'] = df['ghz_fidelity'].fillna(
        (1 - df['cx_error_median'])**2 * (1 - df['readout_error_median'])**3
    )
    
    # Compute Quality Score
    df['quality_score'] = (
        0.35 * df['T2_norm'] +
        0.40 * df['fidelity_norm'] +
        0.25 * df['ghz_fidelity']
    )
    
    return df.sort_values('quality_score', ascending=False)

df_scored = compute_quality_scores(df_backends, ghz_results)
print('Quality scores computed. Top backends:')
cols_display = ['backend', 'n_qubits', 'T2_median_us', 'cx_error_median', 'ghz_fidelity', 'quality_score']
print(df_scored[cols_display].head(10).to_string(index=False))

## Step 6: Generate Backend Report Card

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

def generate_report_card(df_scored, ghz_results, output_path='backend_report_card.png'):
    """Generate a visual Backend Report Card."""
    
    fig = plt.figure(figsize=(16, 10))
    fig.patch.set_facecolor('#F8F9FA')
    
    # Title
    fig.suptitle(
        'IBM Quantum Backend Report Card\nChapter 4 — Applied Quantum Computing',
        fontsize=16, fontweight='bold', y=0.98, color='#1a1a2e'
    )
    
    top_n = min(8, len(df_scored))
    df_plot = df_scored.head(top_n)
    
    # Plot 1: Quality Score Bar Chart
    ax1 = fig.add_subplot(2, 2, 1)
    colors = ['#2563EB' if i == 0 else '#93C5FD' for i in range(len(df_plot))]
    bars = ax1.barh(df_plot['backend'], df_plot['quality_score'], color=colors)
    ax1.set_xlabel('Quality Score (0-1)')
    ax1.set_title('Composite Quality Score\n(Higher = Better)', fontweight='bold')
    ax1.set_xlim(0, 1.05)
    for bar, val in zip(bars, df_plot['quality_score']):
        ax1.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=9)
    ax1.invert_yaxis()
    ax1.set_facecolor('#FFFFFF')
    ax1.grid(axis='x', alpha=0.3)
    
    # Plot 2: T2 Coherence Time
    ax2 = fig.add_subplot(2, 2, 2)
    ax2.barh(df_plot['backend'], df_plot['T2_median_us'], color='#F97316', alpha=0.8)
    ax2.set_xlabel('T₂ Median (μs)')
    ax2.set_title('Coherence Time (T₂)\n(Higher = Better)', fontweight='bold')
    ax2.invert_yaxis()
    ax2.set_facecolor('#FFFFFF')
    ax2.grid(axis='x', alpha=0.3)
    
    # Plot 3: 2Q Gate Error Rate
    ax3 = fig.add_subplot(2, 2, 3)
    err_pct = df_plot['cx_error_median'] * 100
    bar_colors = ['#16A34A' if e < 0.5 else '#EAB308' if e < 1.0 else '#DC2626' for e in err_pct]
    ax3.barh(df_plot['backend'], err_pct, color=bar_colors, alpha=0.8)
    ax3.set_xlabel('2Q Gate Error Rate (%)')
    ax3.set_title('2Q Gate Error Rate\n(Lower = Better)', fontweight='bold')
    ax3.invert_yaxis()
    ax3.set_facecolor('#FFFFFF')
    ax3.grid(axis='x', alpha=0.3)
    green_patch = mpatches.Patch(color='#16A34A', label='<0.5% (Excellent)')
    yellow_patch = mpatches.Patch(color='#EAB308', label='0.5-1.0% (Good)')
    red_patch = mpatches.Patch(color='#DC2626', label='>1.0% (Fair)')
    ax3.legend(handles=[green_patch, yellow_patch, red_patch], fontsize=8, loc='lower right')
    
    # Plot 4: Measured GHZ Fidelity (for tested backends)
    ax4 = fig.add_subplot(2, 2, 4)
    tested_df = df_plot[df_plot['backend'].isin(ghz_results.keys())]
    if len(tested_df) > 0:
        bar_colors4 = ['#2563EB' if f > 0.8 else '#F97316' if f > 0.6 else '#DC2626'
                       for f in tested_df['ghz_fidelity']]
        ax4.barh(tested_df['backend'], tested_df['ghz_fidelity'], color=bar_colors4, alpha=0.8)
        ax4.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='50% (random)')
        ax4.set_xlabel('GHZ Fidelity (fraction ideal outcomes)')
        ax4.set_title('Measured GHZ State Fidelity\n(Higher = Better)', fontweight='bold')
        ax4.set_xlim(0, 1.05)
        ax4.invert_yaxis()
        ax4.legend(fontsize=8)
    else:
        ax4.text(0.5, 0.5, 'No measured results\n(calibration estimate used)',
                ha='center', va='center', transform=ax4.transAxes, fontsize=12, color='gray')
    ax4.set_facecolor('#FFFFFF')
    ax4.grid(axis='x', alpha=0.3)
    
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    # Add timestamp
    fig.text(0.01, 0.01,
             f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M UTC")} | Source: IBM Quantum Calibration Data',
             fontsize=8, color='gray')
    
    plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='#F8F9FA')
    print(f'Report card saved to {output_path}')
    plt.show()

generate_report_card(df_scored, ghz_results)

## Step 7: Print Ranked Report Card Table

In [ ]:
print('='*90)
print('IBM QUANTUM BACKEND REPORT CARD')
print(f'Generated: {datetime.now().strftime("%Y-%m-%d %H:%M UTC")}')
print('='*90)
print()
print('Quality Score = 0.35×(T2/T2_max) + 0.40×(1 - err_2Q/err_max) + 0.25×(GHZ_fidelity)')
print()

report_cols = {
    'backend': 'Backend',
    'n_qubits': 'Qubits',
    'T1_median_us': 'T1(μs)',
    'T2_median_us': 'T2(μs)',
    'cx_error_median': '2Q_Error',
    'readout_error_median': 'Readout_Err',
    'ghz_fidelity': 'GHZ_Fidelity',
    'quality_score': 'Quality_Score'
}

df_report = df_scored[list(report_cols.keys())].rename(columns=report_cols)
df_report = df_report.reset_index(drop=True)
df_report.index = df_report.index + 1  # Start ranking from 1

# Format for display
df_display = df_report.copy()
df_display['T1(μs)'] = df_display['T1(μs)'].round(1)
df_display['T2(μs)'] = df_display['T2(μs)'].round(1)
df_display['2Q_Error'] = df_display['2Q_Error'].apply(lambda x: f'{x:.4f}' if not np.isnan(x) else 'N/A')
df_display['Readout_Err'] = df_display['Readout_Err'].apply(lambda x: f'{x:.4f}' if not np.isnan(x) else 'N/A')
df_display['GHZ_Fidelity'] = df_display['GHZ_Fidelity'].apply(lambda x: f'{x:.3f}' if not np.isnan(x) else 'N/A')
df_display['Quality_Score'] = df_display['Quality_Score'].apply(lambda x: f'{x:.4f}' if not np.isnan(x) else 'N/A')

print(df_display.to_string())
print()
print(f'Total backends evaluated: {len(df_scored)}')
print(f'Backends benchmarked with GHZ circuit: {len(ghz_results)}')
print()

# Top recommendation
top_backend = df_scored.iloc[0]
print(f'TOP RECOMMENDATION: {top_backend["backend"]}')
print(f'  Quality Score: {top_backend["quality_score"]:.4f}')
print(f'  T2 Coherence: {top_backend["T2_median_us"]:.1f} μs')
print(f'  2Q Gate Error: {top_backend["cx_error_median"]:.4f} ({top_backend["cx_error_median"]*100:.2f}%)')
print('='*90)

## Step 8: Your 500-Word Analytical Memo Template

Complete the following memo using your results:

---
**QUANTUM BACKEND SELECTION MEMO**

**To:** [Your Instructor / Course Record]  
**From:** [Your Name]  
**Date:** [Date]  
**Subject:** IBM Quantum Backend Benchmarking Analysis — Chapter 4 Lab

---

**Executive Summary (2 sentences)**

[Which backend ranked highest and why? What was the quality score spread across backends?]

**Quality Score Results**

[Insert your backend ranking table here]

**Analysis — Calibration vs. Measured Performance**

[Paragraph 1: For which backends did calibration specs (T2, 2Q error) predict GHZ fidelity accurately? For which did they not? What might explain discrepancies?]

[Paragraph 2: Were there unexpected results? A backend with worse calibration specs but better measured fidelity? What might explain this?]

[Paragraph 3: What does the spread in quality scores tell you about variation across IBM's backend fleet? Is there a correlation between qubit count and quality?]

**Workload-Specific Recommendation: Pharmaceutical Simulation**

[For a molecular simulation workload requiring 40 qubits and 800 two-qubit gates, which backend would you select? Use Quality Score components — not backend names — as your primary justification. What fidelity can you expect after 800 gates at your backend's median 2Q error rate? Show the calculation.]

**Limitations and Next Steps**

[What are the limitations of the Quality Score formula? What additional metrics would improve it? What would you measure in a more comprehensive benchmark?]

---

*Word count target: 500 words*

In [ ]:
# Bonus: Compute expected fidelity for a 800-gate molecular simulation
print('WORKLOAD FIDELITY PROJECTION: Molecular Simulation (800 2Q gates)')
print('='*65)

for _, row in df_scored.head(5).iterrows():
    if not np.isnan(row['cx_error_median']):
        fidelity_per_gate = 1 - row['cx_error_median']
        expected_fidelity = fidelity_per_gate ** 800
        print(f"{row['backend']:25s} | err={row['cx_error_median']:.4f} | "
              f"F_800gates = {fidelity_per_gate:.4f}^800 = {expected_fidelity:.6f} ({expected_fidelity*100:.4f}%)")

print()
print('Note: Circuits requiring even 1% cumulative fidelity need error correction.')
print('This is why fault tolerance matters for real applications.')